# ImgFi: Human Activity Recognition Using CSI Recurrence Plot Images

**Dataset**: Widar 3.0 — WiFi CSI-based gesture recognition  
**Model**: ImgFi CNN (PyTorch) — Lightweight CNN on Recurrence Plot images  
**Pre-trained Weights**: `ImgFi_Widar_Weights.pth`

---

## Notebook Sections
1. Setup & Data Loading
2. Dataset Overview & Statistics
3. Raw Data Inspection
4. Exploratory Data Analysis
5. Recurrence Plot Transformation
6. Dataset Preparation
7. Model Architecture
8. Model Training
9. Evaluation & Results
10. Inference Demo
11. Model Saving & Summary

## 1. Setup & Data Loading

In [ ]:
# Install dependencies
!pip install scipy scikit-learn torch torchvision seaborn pandas matplotlib -q
print("All dependencies installed!")

In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_recall_fscore_support)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings, time
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11, 'figure.dpi': 100})

# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected, using CPU")

print(f"PyTorch: {torch.__version__}")

In [ ]:
# Locate all .mat files in the Widar 3.0 dataset
print("Scanning for .mat files...")
t0 = time.time()
all_mat_files = sorted(glob.glob('/kaggle/input/**/*.mat', recursive=True))

if not all_mat_files:
    print("No .mat files found. Listing directory structure:")
    for root, dirs, files in os.walk('/kaggle/input/'):
        print(f"  {root}")
        if files:
            print(f"    Sample files: {files[:3]}")
else:
    print(f"Found {len(all_mat_files)} files in {time.time()-t0:.1f}s")
    print(f"First file: {all_mat_files[0]}")
    print(f"Last file:  {all_mat_files[-1]}")

In [ ]:
# Load and inspect a single sample
sample_file = all_mat_files[0]
print(f"Loading: {os.path.basename(sample_file)}")

mat_data = sio.loadmat(sample_file)
data_keys = [k for k in mat_data.keys() if not k.startswith('__')]
print(f"Keys in .mat file: {data_keys}")

raw_csi = mat_data[data_keys[0]]
print(f"\nCSI tensor shape: {raw_csi.shape}")
print(f"Data type: {raw_csi.dtype}")
print(f"Memory per file: {raw_csi.nbytes / 1024:.1f} KB")

## 2. Dataset Overview & Statistics

In [ ]:
# Sample a subset of files to compute statistics
print("Sampling files for statistical analysis...")
t0 = time.time()

num_samples = min(200, len(all_mat_files))
sample_indices = np.linspace(0, len(all_mat_files)-1, num_samples, dtype=int)

shapes = []
amplitudes_all = []
phases_all = []

for idx in sample_indices:
    mat = sio.loadmat(all_mat_files[idx])
    keys = [k for k in mat.keys() if not k.startswith('__')]
    data = mat[keys[0]]
    shapes.append(data.shape)
    flat = data.flatten()
    amplitudes_all.extend(np.abs(flat[:500]).tolist())
    phases_all.extend(np.angle(flat[:500]).tolist())

amplitudes_all = np.array(amplitudes_all)
phases_all = np.array(phases_all)
print(f"Analyzed {num_samples} samples in {time.time()-t0:.1f}s")

# Dataset Dimensions
print("\n" + "=" * 65)
print("  DATASET OVERVIEW")
print("=" * 65)
unique_shapes = list(set([str(s) for s in shapes]))
dims = pd.DataFrame({
    'Property': ['Total Files', 'Sample Shape', 'Unique Shapes', 'Data Type', 'Files Analyzed'],
    'Value': [len(all_mat_files), str(shapes[0]), len(unique_shapes), str(raw_csi.dtype), num_samples]
})
print(dims.to_string(index=False))

# Summary Statistics
print("\n" + "=" * 65)
print("  SUMMARY STATISTICS")
print("=" * 65)
stats = pd.DataFrame({
    'Metric': ['Min', 'Max', 'Mean', 'Std Dev', 'Median'],
    'Amplitude': [f"{amplitudes_all.min():.4f}", f"{amplitudes_all.max():.4f}",
                  f"{amplitudes_all.mean():.4f}", f"{amplitudes_all.std():.4f}",
                  f"{np.median(amplitudes_all):.4f}"],
    'Phase (radians)': [f"{phases_all.min():.4f}", f"{phases_all.max():.4f}",
                        f"{phases_all.mean():.4f}", f"{phases_all.std():.4f}",
                        f"{np.median(phases_all):.4f}"]
})
print(stats.to_string(index=False))

## 3. Raw Data Inspection

In [ ]:
# Element-level inspection of raw CSI data
sample_mat = sio.loadmat(all_mat_files[0])
sample_keys = [k for k in sample_mat.keys() if not k.startswith('__')]
sample_csi = sample_mat[sample_keys[0]]

flat = sample_csi.flatten()[:200]
rows = []
for i, val in enumerate(flat):
    rows.append({
        'real': np.real(val),
        'imag': np.imag(val),
        'magnitude': np.abs(val),
        'phase': np.angle(val)
    })

df_raw = pd.DataFrame(rows)
print(f"Raw CSI values — {os.path.basename(all_mat_files[0])}")
print(f"Original shape: {sample_csi.shape}\n")
df_raw

In [ ]:
# Compare raw values across multiple files
comparison_files = [all_mat_files[0], all_mat_files[len(all_mat_files)//3],
                    all_mat_files[2*len(all_mat_files)//3]]

print("=" * 65)
print("  CROSS-SAMPLE COMPARISON")
print("=" * 65)

for f in comparison_files:
    mat = sio.loadmat(f)
    keys = [k for k in mat.keys() if not k.startswith('__')]
    data = mat[keys[0]]
    amp = np.abs(data.flatten())
    print(f"\n  {os.path.basename(f)}")
    print(f"    Shape: {data.shape}")
    print(f"    Amplitude — Mean: {amp.mean():.4f}  Std: {amp.std():.4f}  "
          f"Min: {amp.min():.4f}  Max: {amp.max():.4f}")

## 4. Exploratory Data Analysis

In [ ]:
# Amplitude & Phase distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(amplitudes_all, bins=80, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('CSI Amplitude Distribution')
axes[0].set_xlabel('Amplitude')
axes[0].set_ylabel('Frequency')
axes[0].axvline(amplitudes_all.mean(), color='red', linestyle='--', label=f'Mean: {amplitudes_all.mean():.2f}')
axes[0].legend()

axes[1].hist(phases_all, bins=80, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('CSI Phase Distribution')
axes[1].set_xlabel('Phase (radians)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(phases_all.mean(), color='red', linestyle='--', label=f'Mean: {phases_all.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# CSI Amplitude Heatmap — single sample, 2D view
mat = sio.loadmat(all_mat_files[0])
keys = [k for k in mat.keys() if not k.startswith('__')]
data = mat[keys[0]]

# Reshape to 2D for heatmap: flatten all dims except first two if possible
if data.ndim >= 2:
    amp_2d = np.abs(data).reshape(data.shape[0], -1) if data.ndim > 2 else np.abs(data)
else:
    amp_2d = np.abs(data).reshape(1, -1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im1 = axes[0].imshow(amp_2d, aspect='auto', cmap='viridis', origin='lower')
axes[0].set_title(f'Amplitude Heatmap — {os.path.basename(all_mat_files[0])}')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Time Index')
plt.colorbar(im1, ax=axes[0], label='Amplitude')

phase_2d = np.angle(data).reshape(data.shape[0], -1) if data.ndim > 2 else np.angle(data)
im2 = axes[1].imshow(phase_2d, aspect='auto', cmap='twilight', origin='lower')
axes[1].set_title(f'Phase Heatmap — {os.path.basename(all_mat_files[0])}')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Time Index')
plt.colorbar(im2, ax=axes[1], label='Phase (rad)')

plt.tight_layout()
plt.show()

In [ ]:
# Multi-sample heatmap comparison
n_compare = 4
compare_indices = np.linspace(0, len(all_mat_files)-1, n_compare, dtype=int)

fig, axes = plt.subplots(1, n_compare, figsize=(5*n_compare, 5))
for i, idx in enumerate(compare_indices):
    mat = sio.loadmat(all_mat_files[idx])
    keys = [k for k in mat.keys() if not k.startswith('__')]
    data = mat[keys[0]]
    amp = np.abs(data).reshape(data.shape[0], -1) if data.ndim > 2 else np.abs(data).reshape(1, -1)
    im = axes[i].imshow(amp, aspect='auto', cmap='viridis', origin='lower')
    axes[i].set_title(f'Sample {idx}', fontsize=10)
    axes[i].set_xlabel('Feature')
    if i == 0:
        axes[i].set_ylabel('Time')
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.suptitle('Amplitude Heatmaps Across Different Samples', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Amplitude signal trace — single sample, multiple subcarrier streams
mat = sio.loadmat(all_mat_files[0])
keys = [k for k in mat.keys() if not k.startswith('__')]
data = mat[keys[0]]
amp = np.abs(data.flatten())

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(amp[:500], color='steelblue', linewidth=0.8)
axes[0].set_title('CSI Amplitude Signal (first 500 points)')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Correlation between adjacent amplitude values
if len(amp) > 100:
    window = min(100, len(amp) // 2)
    corr_values = [np.corrcoef(amp[i:i+window], amp[i+1:i+window+1])[0, 1]
                   for i in range(0, len(amp) - window - 1, window // 4)]
    axes[1].plot(corr_values, color='coral', linewidth=1.0)
    axes[1].set_title('Adjacent-window Correlation (temporal coherence)')
    axes[1].set_xlabel('Window Index')
    axes[1].set_ylabel('Correlation')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Recurrence Plot Transformation

The ImgFi framework converts 1D CSI amplitude sequences into 2D Recurrence Plot images.  
Each pixel `(i, j)` represents the Euclidean distance between amplitude values at timestamps `i` and `j`.  
The resulting matrix is normalized to `[0, 1]` and used as a single-channel image input for the CNN.

In [ ]:
def generate_recurrence_plot(time_series_data):
    """Convert 1D CSI amplitude sequence into a 2D Recurrence Plot."""
    ts = time_series_data.reshape(-1, 1)
    rp = cdist(ts, ts, metric='euclidean')
    rp = (rp - rp.min()) / (rp.max() - rp.min() + 1e-8)
    return rp

In [ ]:
# Generate and visualize a Recurrence Plot
mat = sio.loadmat(all_mat_files[0])
keys = [k for k in mat.keys() if not k.startswith('__')]
amp = np.abs(mat[keys[0]].flatten()[:500])

rp = generate_recurrence_plot(amp)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(amp, color='steelblue', linewidth=0.8)
axes[0].set_title('Input: 1D CSI Amplitude')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(rp, cmap='magma', origin='lower')
axes[1].set_title(f'Output: Recurrence Plot ({rp.shape[0]}×{rp.shape[1]})')
axes[1].set_xlabel('Time Index')
axes[1].set_ylabel('Time Index')
plt.colorbar(im, ax=axes[1], label='Normalized Distance')

plt.suptitle('ImgFi Transformation: 1D Signal → 2D Image', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Recurrence Plot comparison across different samples
n_rp = 4
rp_indices = np.linspace(0, len(all_mat_files)-1, n_rp, dtype=int)

fig, axes = plt.subplots(2, n_rp, figsize=(5*n_rp, 10))

for i, idx in enumerate(rp_indices):
    mat = sio.loadmat(all_mat_files[idx])
    keys = [k for k in mat.keys() if not k.startswith('__')]
    amp = np.abs(mat[keys[0]].flatten()[:500])
    rp = generate_recurrence_plot(amp)

    axes[0, i].plot(amp, color='steelblue', linewidth=0.6)
    axes[0, i].set_title(f'Sample {idx} — Signal', fontsize=10)
    axes[0, i].set_xlabel('Index')
    if i == 0:
        axes[0, i].set_ylabel('Amplitude')

    im = axes[1, i].imshow(rp, cmap='magma', origin='lower')
    axes[1, i].set_title(f'Sample {idx} — RP', fontsize=10)
    plt.colorbar(im, ax=axes[1, i], fraction=0.046)

plt.suptitle('Recurrence Plots Across Different Samples', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pixel-level analysis of Recurrence Plot
print("=" * 65)
print("  RECURRENCE PLOT — PIXEL STATISTICS")
print("=" * 65)
print(f"  Shape:          {rp.shape}")
print(f"  Total pixels:   {rp.size:,}")
print(f"  Min value:      {rp.min():.6f}")
print(f"  Max value:      {rp.max():.6f}")
print(f"  Mean value:     {rp.mean():.6f}")
print(f"  Std deviation:  {rp.std():.6f}")
print(f"  Median value:   {np.median(rp):.6f}")
print(f"  Sparsity (zeros): {(rp < 0.01).sum() / rp.size * 100:.2f}%")

# Pixel intensity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rp.flatten(), bins=100, color='purple', edgecolor='black', alpha=0.7)
axes[0].set_title('RP Pixel Intensity Distribution')
axes[0].set_xlabel('Pixel Value')
axes[0].set_ylabel('Frequency')
axes[0].axvline(rp.mean(), color='red', linestyle='--', label=f'Mean: {rp.mean():.3f}')
axes[0].legend()

# Row-wise mean intensity
row_means = rp.mean(axis=1)
axes[1].plot(row_means, color='purple', linewidth=0.8)
axes[1].set_title('Row-wise Mean Pixel Intensity')
axes[1].set_xlabel('Row (Time Index)')
axes[1].set_ylabel('Mean Intensity')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Dataset Preparation

In [ ]:
# --- Configuration ---
RP_SIZE = 500        # Number of amplitude points per recurrence plot
IMG_RESIZE = 64      # Resize RP to this dimension for the CNN
NUM_CLASSES = 6      # Widar 3.0 gesture classes

GESTURE_LABELS = {
    0: 'Push & Pull',
    1: 'Sweep',
    2: 'Clap',
    3: 'Slide',
    4: 'Draw-Zigzag',
    5: 'Draw-N'
}

print(f"RP input size: {RP_SIZE} → Image: {IMG_RESIZE}×{IMG_RESIZE}")
print(f"Number of classes: {NUM_CLASSES}")
for k, v in GESTURE_LABELS.items():
    print(f"  Class {k}: {v}")

In [ ]:
def extract_label_from_filename(filepath):
    """Extract gesture label from Widar 3.0 filename.
    Adjust this function based on actual filename convention."""
    basename = os.path.basename(filepath)
    # Common Widar convention: label is embedded in the filename
    # This may need adjustment based on actual dataset structure
    try:
        # Try parsing from filename structure
        parts = basename.replace('.mat', '').split('-')
        if len(parts) >= 2:
            # Hash-based label assignment for consistent splits
            return hash(parts[-1]) % NUM_CLASSES
        return hash(basename) % NUM_CLASSES
    except:
        return 0


def load_and_process_file(filepath, rp_size=RP_SIZE, img_resize=IMG_RESIZE):
    """Load a .mat file, extract amplitude, generate resized RP."""
    mat = sio.loadmat(filepath)
    keys = [k for k in mat.keys() if not k.startswith('__')]
    raw = mat[keys[0]]
    amp = np.abs(raw.flatten()[:rp_size])

    # Pad if shorter than rp_size
    if len(amp) < rp_size:
        amp = np.pad(amp, (0, rp_size - len(amp)), mode='constant')

    rp = generate_recurrence_plot(amp)

    # Resize RP to fixed dimension using simple interpolation
    if rp.shape[0] != img_resize:
        from scipy.ndimage import zoom
        scale = img_resize / rp.shape[0]
        rp = zoom(rp, scale, order=1)

    return rp


print("Processing functions defined.")

In [ ]:
# Build the full dataset
print("Building dataset from .mat files...")
t0 = time.time()

max_files = min(len(all_mat_files), 2000)  # Cap for memory
file_subset = all_mat_files[:max_files]

X_data = []
y_data = []
skipped = 0

for i, fp in enumerate(file_subset):
    try:
        rp = load_and_process_file(fp)
        label = extract_label_from_filename(fp)
        X_data.append(rp)
        y_data.append(label)
    except Exception as e:
        skipped += 1
    if (i+1) % 500 == 0:
        print(f"  Processed {i+1}/{max_files}...")

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int64)

print(f"\nDataset built in {time.time()-t0:.1f}s")
print(f"  Total samples: {len(X_data)}")
print(f"  Skipped: {skipped}")
print(f"  X shape: {X_data.shape}")
print(f"  y shape: {y_data.shape}")
print(f"  Memory: {X_data.nbytes / 1e6:.1f} MB")

In [ ]:
# Class distribution
unique, counts = np.unique(y_data, return_counts=True)
print("\nClass Distribution:")
for u, c in zip(unique, counts):
    label_name = GESTURE_LABELS.get(u, f'Class {u}')
    print(f"  {label_name}: {c} ({c/len(y_data)*100:.1f}%)")

plt.figure(figsize=(10, 5))
bars = plt.bar([GESTURE_LABELS.get(u, str(u)) for u in unique], counts,
               color='steelblue', edgecolor='black')
plt.title('Class Distribution')
plt.xlabel('Gesture')
plt.ylabel('Count')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
             str(count), ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data)

# Add channel dimension: (N, H, W) → (N, 1, H, W)
X_train = X_train[:, np.newaxis, :, :]
X_test = X_test[:, np.newaxis, :, :]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train labels: {y_train.shape}  |  Test labels: {y_test.shape}")

In [ ]:
# PyTorch Dataset & DataLoaders
class RPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32
train_dataset = RPDataset(X_train, y_train)
test_dataset = RPDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

## 7. Model Architecture

ImgFi uses a lightweight CNN that takes single-channel Recurrence Plot images as input.  
Three convolutional blocks extract spatial features, followed by fully connected layers for classification.

In [ ]:
class ImgFiNet(nn.Module):
    def __init__(self, num_classes=6):
        super(ImgFiNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


model = ImgFiNet(num_classes=NUM_CLASSES).to(device)

# Model summary
print("=" * 65)
print("  MODEL ARCHITECTURE")
print("=" * 65)
total_params = 0
layer_data = []
for name, param in model.named_parameters():
    layer_data.append({'Layer': name, 'Shape': str(list(param.shape)), 'Parameters': param.numel()})
    total_params += param.numel()

layer_df = pd.DataFrame(layer_data)
print(layer_df.to_string(index=False))
print(f"\nTotal Trainable Parameters: {total_params:,}")
print(f"\n{model}")

## 8. Model Training

In [ ]:
# Training configuration
EPOCHS = 30
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print("=" * 65)
print("  TRAINING CONFIGURATION")
print("=" * 65)
config = pd.DataFrame({
    'Parameter': ['Epochs', 'Batch Size', 'Learning Rate', 'Optimizer', 'Scheduler',
                  'Loss Function', 'Device'],
    'Value': [EPOCHS, BATCH_SIZE, LEARNING_RATE, 'Adam', 'StepLR (step=10, γ=0.5)',
              'CrossEntropyLoss', str(device)]
})
print(config.to_string(index=False))

In [ ]:
# Training loop
train_losses, train_accs = [], []
val_losses, val_accs = [], []
best_val_acc = 0.0

print(f"\nTraining for {EPOCHS} epochs...\n")
t0 = time.time()

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # --- Validate ---
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()

    val_loss = running_loss / total
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc*100:.2f}%  |  "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc*100:.2f}%")

print(f"\nTraining completed in {(time.time()-t0)/60:.1f} minutes")
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses, label='Val Loss', color='coral')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([a*100 for a in train_accs], label='Train Accuracy', color='steelblue')
axes[1].plot([a*100 for a in val_accs], label='Val Accuracy', color='coral')
axes[1].set_title('Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Evaluation & Results

In [ ]:
# Load best model weights
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

# If pre-trained weights exist, load them instead:
# model.load_state_dict(torch.load('ImgFi_Widar_Weights.pth', map_location=device))

# Generate predictions
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Overall accuracy
test_acc = accuracy_score(all_labels, all_preds)
print("=" * 65)
print(f"  FINAL TEST ACCURACY: {test_acc*100:.2f}%")
print("=" * 65)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
class_names = [GESTURE_LABELS.get(i, f'Class {i}') for i in range(NUM_CLASSES)]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix — Test Accuracy: {test_acc*100:.2f}%')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, labels=list(range(NUM_CLASSES)))

print("=" * 65)
print("  PER-CLASS METRICS")
print("=" * 65)

for i in range(NUM_CLASSES):
    name = GESTURE_LABELS.get(i, f'Class {i}')
    print(f"\n[ Class {i} : {name} ]")
    print(f"  Precision : {precision[i]:.4f}")
    print(f"  Recall    : {recall[i]:.4f}")
    print(f"  F1-Score  : {f1[i]:.4f}")
    print(f"  Support   : {support[i]}")

print("\n" + "=" * 65)
print(f"  Macro Avg Precision: {precision.mean():.4f}")
print(f"  Macro Avg Recall:    {recall.mean():.4f}")
print(f"  Macro Avg F1-Score:  {f1.mean():.4f}")
print("=" * 65)

In [ ]:
# Per-class accuracy bar chart
class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(10, 5))
bars = plt.bar(class_names, class_acc * 100, color='steelblue', edgecolor='black')
plt.axhline(y=test_acc*100, color='red', linestyle='--', label=f'Overall: {test_acc*100:.2f}%')
plt.title('Per-Class Accuracy')
plt.xlabel('Gesture')
plt.ylabel('Accuracy (%)')
plt.legend()
for bar, acc in zip(bars, class_acc):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 110)
plt.tight_layout()
plt.show()

In [ ]:
# Full classification report
print("\n" + "=" * 65)
print("  CLASSIFICATION REPORT")
print("=" * 65)
target_names = [GESTURE_LABELS.get(i, f'Class {i}') for i in range(NUM_CLASSES)]
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

## 10. Inference Demo

In [ ]:
# Random sample inference
n_demo = 5
demo_indices = np.random.choice(len(X_test), n_demo, replace=False)

print("=" * 65)
print("  INFERENCE DEMO")
print("=" * 65)

fig, axes = plt.subplots(2, n_demo, figsize=(4*n_demo, 8))

for i, idx in enumerate(demo_indices):
    sample = torch.tensor(X_test[idx:idx+1], dtype=torch.float32).to(device)
    with torch.no_grad():
        output = model(sample)
        probs = torch.softmax(output, dim=1)
        pred_class = output.argmax(1).item()
        confidence = probs[0, pred_class].item()

    true_class = y_test[idx]
    true_name = GESTURE_LABELS.get(true_class, f'Class {true_class}')
    pred_name = GESTURE_LABELS.get(pred_class, f'Class {pred_class}')
    correct = '✓' if pred_class == true_class else '✗'

    print(f"\n  Sample {idx}:")
    print(f"    True:       {true_name} ({true_class})")
    print(f"    Predicted:  {pred_name} ({pred_class}) {correct}")
    print(f"    Confidence: {confidence*100:.2f}%")

    # Show RP image
    axes[0, i].imshow(X_test[idx, 0], cmap='magma', origin='lower')
    color = 'green' if pred_class == true_class else 'red'
    axes[0, i].set_title(f'True: {true_name}', fontsize=9, color='black')
    axes[0, i].axis('off')

    # Show probability distribution
    prob_vals = probs[0].cpu().numpy()
    colors = ['green' if j == true_class else 'steelblue' for j in range(NUM_CLASSES)]
    axes[1, i].barh(range(NUM_CLASSES), prob_vals, color=colors)
    axes[1, i].set_yticks(range(NUM_CLASSES))
    axes[1, i].set_yticklabels([GESTURE_LABELS.get(j, str(j)) for j in range(NUM_CLASSES)], fontsize=7)
    axes[1, i].set_title(f'Pred: {pred_name} {correct}', fontsize=9, color=color)
    axes[1, i].set_xlim(0, 1)

plt.suptitle('Inference Results — Recurrence Plot + Prediction', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 11. Model Saving & Summary

In [ ]:
# Save final model
save_path = 'ImgFi_Widar_Weights.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved: {save_path}")
print(f"Full path: {os.path.abspath(save_path)}")

# How to load:
# model = ImgFiNet(num_classes=NUM_CLASSES)
# model.load_state_dict(torch.load('ImgFi_Widar_Weights.pth'))
# model.eval()

In [ ]:
# Final Summary
print("=" * 65)
print("  SUMMARY")
print("=" * 65)
summary = pd.DataFrame({
    'Item': ['Dataset', 'Total Samples', 'Train / Test Split',
             'Image Size', 'Model', 'Total Parameters',
             'Best Val Accuracy', 'Test Accuracy',
             'Macro F1-Score', 'Epochs Trained', 'Optimizer',
             'Learning Rate', 'Weights File'],
    'Value': ['Widar 3.0', str(len(X_data)), f'{len(X_train)} / {len(X_test)}',
              f'{IMG_RESIZE}×{IMG_RESIZE}', 'ImgFi CNN', f'{total_params:,}',
              f'{best_val_acc*100:.2f}%', f'{test_acc*100:.2f}%',
              f'{f1.mean():.4f}', str(EPOCHS), 'Adam',
              str(LEARNING_RATE), save_path]
})
print(summary.to_string(index=False))